<a href="https://colab.research.google.com/github/Dinesh123reddy/EPAPER_EXTRACTIONS/blob/FEATURE-EXTRACTION/ANDHRA_PRABHA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MAIN CODE**

- **MOUNTING GOOGLE DRIVE**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


- **INSTALLING THE PACKAGES**

In [ ]:
!apt-get install poppler-utils -y # Install system package for pdf2image
!pip install pdf2image # Install the pdf2image python module
!pip install --upgrade pdf2image # Upgrade pdf2image to the latest version
!pip install pymupdf

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  poppler-utils
0 upgraded, 1 newly installed, 0 to remove and 29 not upgraded.
Need to get 186 kB of archives.
After this operation, 696 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 poppler-utils amd64 22.02.0-2ubuntu0.6 [186 kB]
Fetched 186 kB in 0s (397 kB/s)
Selecting previously unselected package poppler-utils.
(Reading database ... 124947 files and directories currently installed.)
Preparing to unpack .../poppler-utils_22.02.0-2ubuntu0.6_amd64.deb ...
Unpacking poppler-utils (22.02.0-2ubuntu0.6) ...
Setting up poppler-utils (22.02.0-2ubuntu0.6) ...
Processing triggers for man-db (2.10.2-1) ...


In [ ]:
pip install --upgrade google-api-python-client google-auth-httplib2 google-auth-oauthlib

In [ ]:
!pip install ultralytics

- **IMPORTING THE LIBRARIES**

In [ ]:
import os
import re
import cv2
import json
import enum
import time
import torch
import shutil
import calendar
import requests
import pandas as pd
from typing import Any
from io import BytesIO
from google import genai
import concurrent.futures
from ultralytics import YOLO
from datetime import datetime
from bs4 import BeautifulSoup
from pydantic import BaseModel
from collections import defaultdict
from pdf2image import convert_from_path
from datetime import datetime, timedelta
from PIL import Image, ImageFilter, ImageEnhance
from pdf2image.exceptions import PDFPageCountError

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  tesseract-ocr-eng tesseract-ocr-osd
The following NEW packages will be installed:
  tesseract-ocr tesseract-ocr-eng tesseract-ocr-osd
0 upgraded, 3 newly installed, 0 to remove and 29 not upgraded.
Need to get 4,816 kB of archives.
After this operation, 15.6 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-eng all 1:4.00~git30-7274cfa-1.1 [1,591 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr-osd all 1:4.00~git30-7274cfa-1.1 [2,990 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 tesseract-ocr amd64 4.1.1-2.1build1 [236 kB]
Fetched 4,816 kB in 2s (2,637 kB/s)
Selecting previously unselected package tesseract-ocr-eng.
(Reading database ... 124977 files and directories currently installed.)
Preparing to unpack .../tesseract-ocr-

- **EXTRACTING THE PAGE URLS FROM CURRENT DATE TO PAST 6 MONTHS**

In [ ]:
def fetch_page_ids_from_api(date_str):
    """
    Fetch PageIDs directly from the API based on conditions.
    """
    url = f'https://epaper.prabhanews.com/Home/GetAllpages?editionid=20&editiondate={date_str}'
    selected_urls = set()

    try:
        response = requests.get(url)
        if response.status_code == 200:
            json_data = response.json()  # Parse JSON response
            total_pages = len(json_data)  # Number of pages available

            if total_pages == 12:
                selected_pages = ["8", "9"]
            else:
                selected_pages = ["6"]

            for item in json_data:
                page_id = item.get("PageId")
                page_number = str(item.get("PageNo"))  # Ensure page number is a string for comparison

                if page_id and page_number in selected_pages:
                    full_url = f'https://epaper.prabhanews.com/Karimnagar?eid=20&edate={date_str}&pgid={page_id}&device=desktop&view=3'

                    if check_url_exists(full_url):
                        selected_urls.add(full_url)
        else:
            print(f"Failed to fetch data for {date_str}. HTTP Status Code: {response.status_code}")

    except Exception as e:
        print(f"An error occurred while fetching data: {e}")

    return selected_urls

def check_url_exists(url):
    """
    Check if a URL exists by sending a HEAD request.
    """
    try:
        response = requests.head(url, allow_redirects=True)
        return response.status_code == 200
    except requests.exceptions.RequestException as e:
        print(f"Error while checking URL: {url}, Exception: {e}")
        return False

if __name__ == "__main__":
    all_files = set()  # Use a set to store unique URLs
    end_date = datetime.today()  # Current date
    start_date = end_date - timedelta(days=30 * 6)  # Go back 6 months

    current_date = end_date
    while current_date >= start_date:
        date_str = current_date.strftime("%d/%m/%Y")  # Format: DD/MM/YYYY
        print(f"Processing date: {date_str}")  # Log progress

        daily_files = fetch_page_ids_from_api(date_str)
        all_files.update(daily_files)  # Add to the set (ensures uniqueness)

        current_date -= timedelta(days=1)  # Move to the previous day

    # Save the URLs to a file
    download_folder = "./"  # Change to the desired folder
    output_file = os.path.join(download_folder, "extracted_links_past_6_months.txt")

    with open(output_file, "w") as file:
        for url in sorted(all_files):  # Sort for consistency
            file.write(url + "\n")

    print("Script execution completed! All extracted links are saved.")

Script execution completed!


['https://epaper.prabhanews.com/Karimnagar?eid=20&edate=01/09/2024&pgid=415723&device=desktop&view=3']

- **FINDING .JPG URLS FROM THE ABOVE URLS**

In [ ]:
# URL of the web page
urls = all_files
matching_files = []
# Fetch the page content
for url in urls:
  response = requests.get(url)

  # Check if the request was successful
  if response.status_code == 200:
      # Get the HTML content
      html_content = response.text
      soup = BeautifulSoup(html_content, 'html.parser')

      # Find all meta tags with property="og:image"
      meta_tags = soup.find_all('meta',property = "og:image")

      # Extract URLs ending with '.ss_jpg'
      image_urls = [tag['content'] for tag in meta_tags if tag.has_attr('content') and tag['content'].endswith('.jpg')]

      # Print the extracted URLs
      if image_urls:
          for Urls in image_urls:
              matching_files.append(Urls)
      else:
          print("No image URLs found matching the criteria.")
  else:
      print(f"Failed to fetch page. Status code: {response.status_code}")

list(set(matching_files))

['https://apfsnew.smartepaper.in/AP/2024/09/01/TlgxKrn/5_06/523f8ccd_06_ss.jpg']

- **CONVERTING _SS.JPG TO HIGH RESOLUTION IMAGES**
- **DOWNLOADING AND MERGING EACH HIGH RESOLUTION IMAGES TO THEIR RESPECTIVE DATES**

In [ ]:
Main_folder = "/content/drive/My Drive/ANDHRA_PRABHA"
if not os.path.exists(Main_folder):
    os.makedirs(Main_folder)

# List of image URLs (Assuming 'image_urls' is populated with URLs)
image_urls = matching_files

# ✅ Function to remove the '_ss' part from URLs
high_res_urls = [url.replace("_ss", "") for url in image_urls]

# ✅ Function to extract year, month, and date from the URL
def extract_date_details(url):
    match = re.search(r"/(\d{4})/(\d{2})/(\d{2})/", url)
    if match:
        year, month, day = match.groups()
        month_name = calendar.month_name[int(month)]  # Convert month number to name
        return year, month_name, f"{year}-{month}-{day}"  # Return year, month name, and formatted date
    return None, None, None

# ✅ Determine the base output folder name
if high_res_urls:
    year, month_name, _ = extract_date_details(high_res_urls[0])
    if year and month_name:
        output_folder = f"ANDHRA_PRABHA-{year}-{month_name.upper()}"
    else:
        output_folder = "ANDHRA_PRABHA_UNKNOWN"
else:
    output_folder = "ANDHRA_PRABHA_UNKNOWN"

# os.makedirs(output_folder, exist_ok=True)  # Ensure the base folder exists
base_folder = os.path.join('/content/drive/MyDrive/ANDHRA_PRABHA', output_folder)
os.makedirs(base_folder, exist_ok=True)

# ✅ Step 1: Group URLs by their date
grouped_urls = defaultdict(list)

for url in high_res_urls:
    _, _, date = extract_date_details(url)
    if date:
        grouped_urls[date].append(url)

# ✅ Step 2: Download images and save them into date-based folders
for date, urls in grouped_urls.items():
    # ✅ Create a sub-folder for this date
    date_folder = os.path.join(base_folder, f"ANDHRA_PRABHA_{date}")
    os.makedirs(date_folder, exist_ok=True)

    for index, url in enumerate(urls):
        try:
            response = requests.get(url)
            if response.status_code == 200:
                # ✅ Open and convert image to RGB format
                img = Image.open(BytesIO(response.content))
                img_rgb = img.convert('RGB')

                # ✅ Save image with an incremented name
                image_filename = f"ANDHRA_PRABHA_{date}_PAGE-{index+1}.jpg"
                image_path = os.path.join(date_folder, image_filename)
                img_rgb.save(image_path, "JPEG")
                print(f"✅ Saved: {image_path}")

            else:
                print(f"❌ Failed to download image from {url}. Status code: {response.status_code}")

        except Exception as e:
            print(f"❌ Error downloading image from {url}: {e}")


✅ Saved: /content/drive/MyDrive/ANDHRA_PRABHA/ANDHRA_PRABHA-2024-SEPTEMBER/ANDHRA_PRABHA_2024-09-01/ANDHRA_PRABHA_2024-09-01_PAGE-1.jpg


- **USING BEST.PT FILE**

In [ ]:
# ✅ Path to the input folder
input_folder = "/content/drive/MyDrive/ANDHRA_PRABHA"

# ✅ Load local YOLO model
model_path = "/content/drive/MyDrive/YOLO FILE/best.pt"  # Update with your actual model path
model = YOLO(model_path)

def process_image(image_path, output_folder, date_folder):
    os.makedirs(output_folder, exist_ok=True)
    original_image_path = os.path.join(output_folder, os.path.basename(image_path))

    # ✅ Save original image only if it doesn't exist
    if not os.path.exists(original_image_path):
        Image.open(image_path).save(original_image_path)

    page_match = re.search(r'Page-(\d+)', image_path, re.IGNORECASE)
    page_number = f"Page-{page_match.group(1)}" if page_match else "Page-1"

    output_articles = [
        f for f in os.listdir(output_folder)
        if f.startswith(f"{date_folder}_{page_number}_article_")
    ]

    # ✅ Skip if all articles already exist
    if output_articles:
        print(f"⏭️ Skipping processing for '{image_path}', articles already extracted.")
        return

    try:
        results = model(image_path)  # Run YOLO only once per image
    except Exception as e:
        print(f"❌ Error processing '{image_path}': {e}")
        return

    original_image = Image.open(image_path)

    for j, box in enumerate(results[0].boxes.xyxy):
        x1, y1, x2, y2 = map(int, box)

        if x2 - x1 <= 0 or y2 - y1 <= 0:
            print(f"⚠️ Skipping invalid crop for {image_path}, prediction {j + 1}")
            continue

        article_filename = f"{date_folder}_{page_number}_article_{j + 1}"
        article_image_path = os.path.join(output_folder, f"{article_filename}.jpg")

        # ✅ Check if article image already exists
        if os.path.exists(article_image_path):
            print(f"⏭️ Skipping existing article '{article_filename}.jpg'")
            continue

        cropped_image = original_image.crop((x1, y1, x2, y2))
        cropped_image.save(article_image_path)
        print(f"✅ Saved article '{article_filename}.jpg'")

for month_folder in os.listdir(input_folder):
    month_folder_path = os.path.join(input_folder, month_folder)
    if not os.path.isdir(month_folder_path):
        continue

    for date_folder in os.listdir(month_folder_path):
        date_folder_path = os.path.join(month_folder_path, date_folder)
        if not os.path.isdir(date_folder_path):
            continue

        print(f"📂 Processing date folder: {date_folder}")
        output_date_folder = os.path.join(input_folder, month_folder, date_folder)
        os.makedirs(output_date_folder, exist_ok=True)

        image_files = [os.path.join(date_folder_path, f) for f in os.listdir(date_folder_path) if f.lower().endswith(('.jpg', '.png', '.jpeg'))]

        with concurrent.futures.ThreadPoolExecutor(max_workers=10) as executor:
            executor.map(lambda img: process_image(img, output_date_folder, date_folder), image_files)

print("🎉 Article extraction completed!")

📂 Processing date folder: ANDHRA_PRABHA_2024-09-01

image 1/1 /content/drive/MyDrive/ANDHRA_PRABHA/ANDHRA_PRABHA-2024-SEPTEMBER/ANDHRA_PRABHA_2024-09-01/ANDHRA_PRABHA_2024-09-01_PAGE-1.jpg: 416x640 11 ARTICLEs, 304.4ms
Speed: 3.7ms preprocess, 304.4ms inference, 0.9ms postprocess per image at shape (1, 3, 416, 640)
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_1.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_2.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_3.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_4.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_5.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_6.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_7.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_8.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_9.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_Page-1_article_10.jpg'
✅ Saved article 'ANDHRA_PRABHA_2024-09-01_

- **DELETING ORIGINAL IMAGES FROM THE FOLDER**

In [ ]:
deleted_count = 0

# Walk through folders and delete files that do not contain "article" in the filename
for root, _, files in os.walk(input_folder):
    for filename in files:
        if "article" not in filename.lower():  # Check if "article" is not in the filename
            file_path = os.path.join(root, filename)
            try:
                os.remove(file_path)
                deleted_count += 1
                print(f"Deleted: {file_path}")
            except Exception as e:
                print(f"Error deleting {file_path}: {e}")

print(f"✅ Done. Deleted {deleted_count} file(s) that did not contain 'article' in the filename.")

- **APPENDING ALL ARTICLES TO A LIST**

In [ ]:
def get_images_in_drive_folder(root_folder):
    image_files = []
    # Walk through all subdirectories and files in the folder
    for subdir, _, files in os.walk(root_folder):
        for file in files:
            # Check if the file is an image (you can add more extensions if needed)
            if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')):
                # Add the full path of the image file
                image_files.append(os.path.join(subdir, file))
    return image_files

image_files = get_images_in_drive_folder(input_folder)

# To find the length of image files
len(image_files)

- **INITIALIZING SCENE TYPES AND FEATURES**

In [ ]:
class NewsTypes(enum.Enum):
  murder = 'murder'
  assault = 'assault'
  robbery = 'robbery'
  custodial_deaths = 'custodial deaths'
  sexual_harassment ='sexual harassment'
  kidnapping = 'kidnapping'
  homicide = 'homicide'
  stabbing = 'stabbing'
  shooting = 'shooting'
  theft = 'theft'
  burglary = 'burglary'
  vehicle_theft = 'vehicle theft'
  vandalism = 'vandalism'
  larceny = 'larceny'
  housebreaking = 'housebreaking'
  online_fraud = 'online fraud'
  hacking = 'hacking'
  identity_theft = 'identity theft'
  cyberstalking = 'cyberstalking'
  phishing = 'phishing'
  ransomware = 'ransomware'
  cyber_attack = 'cyber attack'
  drug_trafficking = 'drug trafficking'
  drug_possession = 'drug possession'
  drug_distribution = 'drug distribution'
  narcotics = 'narcotics'
  smuggling = 'smuggling'
  drug_seizure = 'drug seizure'
  child_abuse = 'child abuse'
  racketeering = 'racketeering'
  syndicate = 'syndicate'
  extortion = 'extortion'
  fraud = 'fraud'
  embezzlement = 'embezzlement'
  corruption = 'corruption'
  money_laundering = 'money laundering'
  insider_trading = 'insider trading'
  human_trafficking = 'human trafficking'
  road_accident = 'road accident'
  cyberbullying = 'cyberbullying'
  fatal_accident = 'fatal accident'
  others = 'others'
  protest = 'protest'
  religious_conflict = 'religious conflict'
  riot = 'riot'
  mob_violence = 'mob violence'
  public_disturbance = 'public disturbance'
  mob_attack = 'mob attack'
  Election_campaign = 'Election campaign'
  political_party = 'political party'
  policy_announcement = 'policy announcement'
  misuse_of_funds = 'misuse of funds'
  abuse_of_power = 'abuse of power'
  accident = 'accident'
  natural_disaster = 'natural disaster'
  court_cases = 'court cases'
  judgement = 'judgement'
  legal_proceedings = 'legal proceedings'
  arrest = 'arrest'
  health_hazard = 'health hazard'
  medical_malpractice = 'medical malpractice'
  infrastructure_collapse = 'infrastructure collapse'
  utility_failure = 'utility failure'
  child_labor = 'child labor'
  domestic_violence = 'domestic violence'
  #womens_safety = 'womens safety'
  municipal_issues = 'municipal issues'
  local_development = 'local development'
  village_news = 'village news'
  exam_news = 'exam news'
  social_awareness= 'social awareness'
  farmer_issues = 'farmer issues'
  awards_and_achivements = 'awards and achievements'
  inspection = 'inspection'
  employeement = 'employeement'
  party_switching = 'party switching'
  sports_event = 'sports event'
  festival = 'festival'
  cultural_program = 'cultural program'
  sympathy_condolence = 'Sympathy and Condolence'

class Person(BaseModel):
  name: str
  age: str
  location: str

class InvolvedPersons(BaseModel):
  name: str
  role: str

class CommonFeatures(BaseModel):
  headline: str
  date_time: str
  day_of_week: str
  source: str
  location: str
  main_subject: str
  keywords: list[str]
  summary: str
  tone_of_news: str
  impact_and_significance: str
  quotes_and_statements: list[str]
  # supporting_data: str
  public_reaction: str
  references_to_past_events: str
  images_and_media: str
  conclusion_and_future_implications: str

class NewsFeatures(BaseModel):
  News_Type: NewsTypes
  victim: Person
  accused: Person
  suspect: Person
  witness: Person
  Involved_persons: InvolvedPersons
  Common_Features: CommonFeatures
  News_Specific_Features: list[str]

- **EXTRACTING FEATURES OF EACH MONTH AND SAVING AS .JSON FILES IN THEIR RESPECTIVE MONTH FOLDERS**

In [ ]:
# Google Gemini API client
client = genai.Client(api_key="GEMINI_API_KEY")

# Google Maps API Key
API_KEY = "GOOGLE_MAPS_API_KEY"

# Regular expression to extract the month name (YYYY-MONTH) from file path
month_pattern = re.compile(r"ANDHRA_PRABHA-(\d{4})-(\w+)")

# Extract the month from the first file's path
match = month_pattern.search(image_files[0])
if match:
    extracted_month = match.group(2).upper()  # Extract and convert to uppercase (e.g., "NOVEMBER")
else:
    raise ValueError("Could not extract month from file path!")

# Set output directory and filename based on extracted month
output_dir = f"{extracted_month}_json"
output_filename = os.path.join(output_dir, f"{extracted_month}.json")

# Create the output directory if it does not exist
os.makedirs(output_dir, exist_ok=True)

# Processing configuration
feature_extraction = "Translate to English and if features are not available then keep it N/A"
batch_size = 10

# Regular expression pattern to match YYYY-MM-DD format
date_pattern = re.compile(r"(\d{4}-\d{2}-\d{2})")

def add_date_feature(path, response):
    file_name = os.path.basename(path)  # Extract filename from full path
    date_match = date_pattern.search(file_name)  # Search for date in filename

    if date_match:
        try:
            published_date = datetime.strptime(date_match.group(0), "%Y-%m-%d").date()
            response['Published_Date'] = str(published_date)
        except ValueError:
            response['Published_Date'] = "Invalid Date Format"
    else:
        response['Published_Date'] = "N/A"

    response['Source_file'] = file_name
    return response

# Function to process image with Gemini API
def vision_gemini(image_path):
    response = client.models.generate_content(
        model="gemini-2.0-flash",
        contents=[feature_extraction, image_path],
        config={
            'response_mime_type': 'application/json',
        },
    )
    return response.text

# Function to get latitude and longitude
def get_geolocation(location_text):
    url = f"https://maps.googleapis.com/maps/api/geocode/json?address={location_text}&key={API_KEY}"
    geo_response = requests.get(url).json()
    if geo_response["status"] == "OK":
        return f"{geo_response['results'][0]['geometry']['location']['lat']},{geo_response['results'][0]['geometry']['location']['lng']}"
    return "Unknown Location"

# Processing image files
all_results = []

for index, image_path in enumerate(image_files):
    if index > 0 and index % batch_size == 0:
        print("Rate limit reached. Waiting for 10 seconds...")
        time.sleep(10)

    json_filename = os.path.join(output_dir, f"{os.path.basename(image_path)}.json")

    image = Image.open(image_path)
    response = vision_gemini(image)

    try:
        json_data = json.loads(response)
        json_data = add_date_feature(image_path, json_data[0] if isinstance(json_data, list) and json_data else json_data)

        if 'Common_Features' in json_data and 'location' in json_data['Common_Features']:
            json_data['Common_Features']['location'] = get_geolocation(json_data['Common_Features']['location'])

        # Save JSON locally
        with open(json_filename, "w", encoding="utf-8") as json_file:
            json.dump(json_data, json_file, indent=4, ensure_ascii=False)

        all_results.append(json_data)
        print(f"Processed {image_path} and saved JSON.")
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON for image {image_path}: {e}")

# Save all results to a single JSON file
with open(output_filename, "w", encoding="utf-8") as json_file:
    json.dump(all_results, json_file, indent=4, ensure_ascii=False)

print(f"Successfully saved all results to {output_filename}")

- **ADDING JSON FILES TO GOOGLE DRIVE**

In [ ]:
# Define the Google Drive path where you want to save the file
drive_path = "/content/drive/My Drive/andhra-prabha"
if not os.path.exists(drive_path):
    os.makedirs(drive_path)

# Move the JSON file from local storage to Google Drive
shutil.copy(output_filename, drive_path)

# Extract the JSON file name from the output filename (e.g., "NOVEMBER.json")
json_file_name = os.path.basename(output_filename)

print(f"Successfully saved {json_file_name} to Google Drive at {drive_path}")

- **CONVERTING IN TO .CSV FORMAT WITH SEPARATING FEATURE ENTITIES INTO SEPARATE COLUMNS**

In [ ]:
# Step 2: Define JSON file paths in Google Drive
json_dir = "/content/drive/MyDrive/andhra-prabha"

# List to store all flattened rows
flattened_data_list = []

json_files = [os.path.join(json_dir, filename) for filename in os.listdir(json_dir) if filename.endswith(".json")]

for json_file in json_files:
    if not os.path.exists(json_file):
        print(f"Skipping {json_file}: File does not exist.")
        continue  # Skip missing files

    # Read the JSON file
    with open(json_file, "r", encoding="utf-8") as f:
        try:
            json_data = json.load(f)  # Load the JSON data
        except json.JSONDecodeError as e:
            print(f"Error reading {json_file}: {e}")
            continue

    # If the JSON data is a list of dictionaries, process each entry
    if isinstance(json_data, list):
        for doc in json_data:
            flattened_data = {
                "Source_file": doc.get("Source_file", "N/A"),
                "Published_Date": doc.get("Published_Date", "N/A"),
                "News_Type": doc.get("News_Type", "N/A"),
            }

            # Handle dictionary fields with prefixes
            for category in ['Involved_persons', 'victim', 'witness', 'suspect', 'accused']:
                if category in doc and isinstance(doc[category], dict):
                    for key, value in doc[category].items():
                        flattened_data[f"{category}_{key}"] = value

            # Handle News_Specific_Features as a semicolon-separated string
            flattened_data["News_Specific_Features"] = "; ".join(doc.get("News_Specific_Features", []))

            # Handle Common_Features with "Common_Features_" prefix
            if "Common_Features" in doc and isinstance(doc["Common_Features"], dict):
                for key, value in doc["Common_Features"].items():
                    flattened_data[f"Common_Features_{key}"] = "; ".join(value) if isinstance(value, list) else value

            # Append to the list
            flattened_data_list.append(flattened_data)

    else:
        print(f"Skipping {json_file}: Data is not in the expected list format.")

# Convert to DataFrame
df = pd.DataFrame(flattened_data_list)

# Save to CSV only if there is data
if not df.empty:
    csv_filename = json_dir + "/ANDHRA_PRABHA_FULL.csv"
    df.to_csv(csv_filename, index=False)
    print(f"Successfully saved data to {csv_filename}")
else:
    print("No valid data found. CSV file not created.")